# Comprehensive WordNet Analysis: Gemma vs Vault Gemma

This notebook performs a complete analysis of geometric representations for:
- **Models**: Gemma-2b and Vault Gemma
- **Datasets**: WordNet Noun Synsets and WordNet Verb Synsets
- **Metrics**: Intra-Class Variance and Hierarchical Orthogonality

Based on the paper: "THE GEOMETRY OF CATEGORICAL AND HIERARCHICAL CONCEPTS IN LARGE LANGUAGE MODELS"

In [ ]:
import torch
import numpy as np
import json
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
from scipy import stats
import hierarchical as hrc
import random

# Set plotting style
sns.set_theme(context="paper", style="whitegrid", palette="colorblind", font_scale=1.5)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(100)
np.random.seed(100)
random.seed(100)

## 1. Configuration

In [ ]:
# Model configurations
GEMMA_MODEL = "google/gemma-2b"
VAULT_GEMMA_MODEL = "vault-ai/gemma-2b"  # UPDATE THIS to your Vault Gemma model path

# Datasets to analyze
DATASETS = ['noun', 'verb']

# Model name for data files
MODEL_DATA_NAME = 'gemma'  # Data files are named with 'gemma' suffix

# Data path configuration
# Default: 'data' (relative to notebook location)
# Custom: '/absolute/path/to/your/data' or 'relative/path/to/data'
DATA_PATH = 'data'  # UPDATE THIS to your custom data path if needed

print(f"Gemma Model: {GEMMA_MODEL}")
print(f"Vault Gemma Model: {VAULT_GEMMA_MODEL}")
print(f"Datasets: {DATASETS}")
print(f"Data Path: {DATA_PATH}")

## 2. Analysis Function Definitions

In [ ]:
def analyze_model_dataset(model_name, dataset_type, g, vocab_dict, vocab_list, data_path='data'):
    """
    Analyze a single model-dataset combination.
    
    Args:
        model_name: Name of the model (for identification)
        dataset_type: 'noun' or 'verb'
        g: Whitened unembedding matrix
        vocab_dict: Vocabulary dictionary
        vocab_list: Vocabulary list
        data_path: Path to data directory (default: 'data')
    
    Returns:
        Dictionary with analysis results
    """
    print(f"\n{'='*80}")
    print(f"Analyzing: {model_name} - {dataset_type.upper()} dataset")
    print(f"{'='*80}")
    
    # Load categories and hierarchy with custom data path
    cats, G, sorted_keys = hrc.get_categories(dataset_type, MODEL_DATA_NAME, data_path=data_path)
    print(f"Loaded {len(sorted_keys)} categories from {data_path}")
    
    # Estimate concept vectors
    concept_vectors = {}
    for node in tqdm(sorted_keys, desc="Computing concept vectors"):
        lemmas = cats[node]
        concept_vectors[node] = hrc.estimate_cat_dir(lemmas, g, vocab_dict)
    
    # Compute projection statistics (intra-class variance)
    projection_stats = {}
    for node in sorted_keys:
        lda_dir = concept_vectors[node]['lda']
        lda_dir_normalized = lda_dir / lda_dir.norm()
        
        # Get embeddings for words in this category
        category_indices = hrc.category_to_indices(cats[node], vocab_dict)
        category_embeddings = g[category_indices]
        
        # Compute projections
        projections = category_embeddings @ lda_dir_normalized
        projections_np = projections.cpu().numpy()
        
        projection_stats[node] = {
            'projections': projections_np,
            'mean': np.mean(projections_np),
            'std': np.std(projections_np),
            'variance': np.var(projections_np),
            'num_words': len(projections_np)
        }
    
    # Compute hierarchical orthogonality
    hierarchical_stats = {}
    for node in sorted_keys:
        predecessors = list(G.predecessors(node))
        if len(predecessors) > 0:
            parent = predecessors[0]
            
            # Get vectors
            l_child = concept_vectors[node]['lda']
            l_parent = concept_vectors[parent]['lda']
            
            # Normalize
            l_child_norm = l_child / l_child.norm()
            l_parent_norm = l_parent / l_parent.norm()
            
            # Compute difference vector
            d = l_child - l_parent
            d_norm = d / d.norm()
            
            # Compute cosine similarity
            cos_parent_diff = (l_parent_norm @ d_norm).item()
            cos_parent_child = (l_parent_norm @ l_child_norm).item()
            
            hierarchical_stats[node] = {
                'parent': parent,
                'cos_parent_diff': cos_parent_diff,
                'cos_parent_child': cos_parent_child
            }
    
    # Compute summary statistics
    avg_std = np.mean([stats['std'] for stats in projection_stats.values()])
    avg_variance = np.mean([stats['variance'] for stats in projection_stats.values()])
    avg_ortho = np.mean([abs(stats['cos_parent_diff']) for stats in hierarchical_stats.values()])
    
    print(f"\nSummary Statistics:")
    print(f"  Average Std Dev: {avg_std:.6f}")
    print(f"  Average Variance: {avg_variance:.6f}")
    print(f"  Average |cos(parent, diff)|: {avg_ortho:.6f}")
    print(f"  Number of hierarchical relationships: {len(hierarchical_stats)}")
    
    return {
        'model_name': model_name,
        'dataset_type': dataset_type,
        'categories': sorted_keys,
        'cats': cats,
        'graph': G,
        'concept_vectors': concept_vectors,
        'projection_stats': projection_stats,
        'hierarchical_stats': hierarchical_stats,
        'summary': {
            'avg_std': avg_std,
            'avg_variance': avg_variance,
            'avg_ortho': avg_ortho
        }
    }

## 3. Load Gemma Model and Analyze

In [ ]:
print("="*80)
print("LOADING GEMMA MODEL")
print("="*80)

# Load Gemma model
g_gemma, _, _ = hrc.get_g(GEMMA_MODEL, device)
vocab_dict_gemma, vocab_list_gemma = hrc.get_vocab(GEMMA_MODEL)

print(f"Vocabulary size: {len(vocab_list_gemma)}")
print(f"Embedding dimension: {g_gemma.shape[1]}")

In [ ]:
# Analyze Gemma on both datasets
gemma_results = {}

for dataset in DATASETS:
    gemma_results[dataset] = analyze_model_dataset(
        model_name="Gemma",
        dataset_type=dataset,
        g=g_gemma,
        vocab_dict=vocab_dict_gemma,
        vocab_list=vocab_list_gemma,
        data_path=DATA_PATH
    )

print("\n✓ Gemma analysis complete")

## 4. Load Vault Gemma Model and Analyze

In [ ]:
print("="*80)
print("LOADING VAULT GEMMA MODEL")
print("="*80)

# Load Vault Gemma model
g_vault, _, _ = hrc.get_g(VAULT_GEMMA_MODEL, device)
vocab_dict_vault, vocab_list_vault = hrc.get_vocab(VAULT_GEMMA_MODEL)

print(f"Vocabulary size: {len(vocab_list_vault)}")
print(f"Embedding dimension: {g_vault.shape[1]}")

In [ ]:
# Analyze Vault Gemma on both datasets
vault_results = {}

for dataset in DATASETS:
    vault_results[dataset] = analyze_model_dataset(
        model_name="Vault Gemma",
        dataset_type=dataset,
        g=g_vault,
        vocab_dict=vocab_dict_vault,
        vocab_list=vocab_list_vault,
        data_path=DATA_PATH
    )

print("\n✓ Vault Gemma analysis complete")

## 5. Comparison and Evaluation

### 5.1 Metric 1: Intra-Class Variance Comparison

In [ ]:
def compare_intra_class_variance(gemma_res, vault_res, dataset_type):
    """
    Compare intra-class variance between Gemma and Vault Gemma.
    """
    print(f"\n{'='*80}")
    print(f"INTRA-CLASS VARIANCE COMPARISON - {dataset_type.upper()}")
    print(f"{'='*80}")
    
    categories = gemma_res['categories']
    
    gemma_stds = [gemma_res['projection_stats'][cat]['std'] for cat in categories]
    vault_stds = [vault_res['projection_stats'][cat]['std'] for cat in categories]
    
    mean_gemma_std = np.mean(gemma_stds)
    mean_vault_std = np.mean(vault_stds)
    ratio = mean_vault_std / mean_gemma_std if mean_gemma_std > 0 else float('inf')
    
    print(f"\nAverage σ_gemma: {mean_gemma_std:.6f}")
    print(f"Average σ_vault:  {mean_vault_std:.6f}")
    print(f"Ratio (σ_vault/σ_gemma): {ratio:.4f}\n")
    
    # Statistical test
    gemma_vars = [gemma_res['projection_stats'][cat]['variance'] for cat in categories]
    vault_vars = [vault_res['projection_stats'][cat]['variance'] for cat in categories]
    
    t_stat, p_value = stats.ttest_rel(vault_vars, gemma_vars)
    print(f"Paired t-test: t={t_stat:.4f}, p={p_value:.6f}")
    
    if p_value < 0.05 and t_stat > 0:
        print("✓ Vault Gemma has SIGNIFICANTLY HIGHER variance (p < 0.05)")
    elif p_value < 0.05 and t_stat < 0:
        print("✗ Vault Gemma has SIGNIFICANTLY LOWER variance (p < 0.05)")
    else:
        print("~ No significant difference (p >= 0.05)")
    
    return {
        'gemma_stds': gemma_stds,
        'vault_stds': vault_stds,
        'mean_gemma_std': mean_gemma_std,
        'mean_vault_std': mean_vault_std,
        'ratio': ratio,
        't_stat': t_stat,
        'p_value': p_value
    }

# Compare for both datasets
variance_comparisons = {}
for dataset in DATASETS:
    variance_comparisons[dataset] = compare_intra_class_variance(
        gemma_results[dataset],
        vault_results[dataset],
        dataset
    )

### 5.2 Metric 2: Hierarchical Orthogonality Comparison

In [ ]:
def compare_hierarchical_orthogonality(gemma_res, vault_res, dataset_type):
    """
    Compare hierarchical orthogonality between Gemma and Vault Gemma.
    """
    print(f"\n{'='*80}")
    print(f"HIERARCHICAL ORTHOGONALITY COMPARISON - {dataset_type.upper()}")
    print(f"{'='*80}")
    
    # Get nodes with hierarchical relationships
    nodes_with_parents = list(gemma_res['hierarchical_stats'].keys())
    
    gemma_cos = [gemma_res['hierarchical_stats'][node]['cos_parent_diff'] 
                 for node in nodes_with_parents]
    vault_cos = [vault_res['hierarchical_stats'][node]['cos_parent_diff'] 
                 for node in nodes_with_parents]
    
    mean_gemma_ortho = np.mean(np.abs(gemma_cos))
    mean_vault_ortho = np.mean(np.abs(vault_cos))
    
    print(f"\nAverage |cos(parent, diff)|_gemma: {mean_gemma_ortho:.6f}")
    print(f"Average |cos(parent, diff)|_vault:  {mean_vault_ortho:.6f}")
    print(f"Difference: {mean_vault_ortho - mean_gemma_ortho:.6f}\n")
    
    # Statistical test
    abs_gemma = np.abs(gemma_cos)
    abs_vault = np.abs(vault_cos)
    
    t_stat, p_value = stats.ttest_rel(abs_vault, abs_gemma)
    print(f"Paired t-test: t={t_stat:.4f}, p={p_value:.6f}")
    
    if p_value < 0.05 and t_stat < 0:
        print("✓ Vault Gemma has SIGNIFICANTLY BETTER orthogonality (p < 0.05)")
    elif p_value < 0.05 and t_stat > 0:
        print("✗ Vault Gemma has SIGNIFICANTLY WORSE orthogonality (p < 0.05)")
    else:
        print("~ No significant difference (p >= 0.05)")
    
    return {
        'nodes': nodes_with_parents,
        'gemma_cos': gemma_cos,
        'vault_cos': vault_cos,
        'mean_gemma_ortho': mean_gemma_ortho,
        'mean_vault_ortho': mean_vault_ortho,
        't_stat': t_stat,
        'p_value': p_value
    }

# Compare for both datasets
orthogonality_comparisons = {}
for dataset in DATASETS:
    orthogonality_comparisons[dataset] = compare_hierarchical_orthogonality(
        gemma_results[dataset],
        vault_results[dataset],
        dataset
    )

## 6. Visualizations

### 6.1 Intra-Class Variance Visualization

In [ ]:
# Create comprehensive variance comparison plot
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for idx, dataset in enumerate(DATASETS):
    comp = variance_comparisons[dataset]
    categories = gemma_results[dataset]['categories'][:20]  # Top 20 for readability
    
    gemma_stds_subset = comp['gemma_stds'][:20]
    vault_stds_subset = comp['vault_stds'][:20]
    
    x = np.arange(len(categories))
    width = 0.35
    
    # Standard deviation comparison
    axes[idx, 0].bar(x - width/2, gemma_stds_subset, width, label='Gemma', alpha=0.8, color='blue')
    axes[idx, 0].bar(x + width/2, vault_stds_subset, width, label='Vault Gemma', alpha=0.8, color='green')
    axes[idx, 0].set_xlabel('Category (top 20)')
    axes[idx, 0].set_ylabel('Standard Deviation (σ)')
    axes[idx, 0].set_title(f'{dataset.upper()}: Intra-Class Variance\nRatio={comp["ratio"]:.3f}, p={comp["p_value"]:.4f}')
    axes[idx, 0].set_xticks(x)
    axes[idx, 0].set_xticklabels(categories, rotation=45, ha='right', fontsize=8)
    axes[idx, 0].legend()
    axes[idx, 0].grid(axis='y', alpha=0.3)
    
    # Ratio plot
    ratios = [v/g if g > 0 else 0 for v, g in zip(comp['vault_stds'], comp['gemma_stds'])]
    colors = ['green' if r > 1 else 'blue' for r in ratios[:20]]
    axes[idx, 1].bar(x, ratios[:20], alpha=0.8, color=colors)
    axes[idx, 1].axhline(y=1, color='red', linestyle='--', linewidth=2, label='Equal')
    axes[idx, 1].set_xlabel('Category (top 20)')
    axes[idx, 1].set_ylabel('Ratio (σ_vault / σ_gemma)')
    axes[idx, 1].set_title(f'{dataset.upper()}: Variance Ratio\n(>1 = Vault higher variance)')
    axes[idx, 1].set_xticks(x)
    axes[idx, 1].set_xticklabels(categories, rotation=45, ha='right', fontsize=8)
    axes[idx, 1].legend()
    axes[idx, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wordnet_intra_class_variance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: wordnet_intra_class_variance_comparison.png")

### 6.2 Hierarchical Orthogonality Visualization

In [ ]:
# Create comprehensive orthogonality comparison plot
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for idx, dataset in enumerate(DATASETS):
    comp = orthogonality_comparisons[dataset]
    nodes_subset = comp['nodes'][:20]  # Top 20 for readability
    
    gemma_cos_subset = comp['gemma_cos'][:20]
    vault_cos_subset = comp['vault_cos'][:20]
    
    x = np.arange(len(nodes_subset))
    width = 0.35
    
    # Raw cosine similarity
    axes[idx, 0].bar(x - width/2, gemma_cos_subset, width, label='Gemma', alpha=0.8, color='blue')
    axes[idx, 0].bar(x + width/2, vault_cos_subset, width, label='Vault Gemma', alpha=0.8, color='green')
    axes[idx, 0].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    axes[idx, 0].set_xlabel('Category (top 20)')
    axes[idx, 0].set_ylabel('cos(l_parent, d)')
    axes[idx, 0].set_title(f'{dataset.upper()}: Hierarchical Orthogonality\np={comp["p_value"]:.4f}')
    axes[idx, 0].set_xticks(x)
    axes[idx, 0].set_xticklabels(nodes_subset, rotation=45, ha='right', fontsize=8)
    axes[idx, 0].legend()
    axes[idx, 0].grid(axis='y', alpha=0.3)
    
    # Absolute values (lower is better)
    abs_gemma = np.abs(gemma_cos_subset)
    abs_vault = np.abs(vault_cos_subset)
    axes[idx, 1].bar(x - width/2, abs_gemma, width, label='Gemma', alpha=0.8, color='blue')
    axes[idx, 1].bar(x + width/2, abs_vault, width, label='Vault Gemma', alpha=0.8, color='green')
    axes[idx, 1].set_xlabel('Category (top 20)')
    axes[idx, 1].set_ylabel('|cos(l_parent, d)|')
    axes[idx, 1].set_title(f'{dataset.upper()}: Absolute Orthogonality\n(Lower = Better)')
    axes[idx, 1].set_xticks(x)
    axes[idx, 1].set_xticklabels(nodes_subset, rotation=45, ha='right', fontsize=8)
    axes[idx, 1].legend()
    axes[idx, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wordnet_hierarchical_orthogonality_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: wordnet_hierarchical_orthogonality_comparison.png")

### 6.3 Summary Comparison Across Datasets

In [ ]:
# Create summary comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets_labels = [d.upper() for d in DATASETS]
x = np.arange(len(DATASETS))
width = 0.35

# Plot 1: Average variance ratio by dataset
variance_ratios = [variance_comparisons[d]['ratio'] for d in DATASETS]
colors_var = ['green' if r > 1 else 'blue' for r in variance_ratios]
axes[0].bar(x, variance_ratios, alpha=0.8, color=colors_var)
axes[0].axhline(y=1, color='red', linestyle='--', linewidth=2, label='Equal Variance')
axes[0].set_xlabel('Dataset')
axes[0].set_ylabel('Variance Ratio (σ_vault / σ_gemma)')
axes[0].set_title('Intra-Class Variance Ratio by Dataset\n(>1 = Vault has higher variance)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(datasets_labels)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(variance_ratios):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Plot 2: Average orthogonality by dataset
gemma_orthos = [orthogonality_comparisons[d]['mean_gemma_ortho'] for d in DATASETS]
vault_orthos = [orthogonality_comparisons[d]['mean_vault_ortho'] for d in DATASETS]

axes[1].bar(x - width/2, gemma_orthos, width, label='Gemma', alpha=0.8, color='blue')
axes[1].bar(x + width/2, vault_orthos, width, label='Vault Gemma', alpha=0.8, color='green')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Average |cos(l_parent, d)|')
axes[1].set_title('Hierarchical Orthogonality by Dataset\n(Lower = Better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(datasets_labels)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wordnet_summary_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: wordnet_summary_comparison.png")

## 7. Final Summary Report

In [ ]:
print("="*80)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("="*80)

for dataset in DATASETS:
    print(f"\n{'='*80}")
    print(f"{dataset.upper()} DATASET")
    print(f"{'='*80}")
    
    var_comp = variance_comparisons[dataset]
    ortho_comp = orthogonality_comparisons[dataset]
    
    print(f"\n1. INTRA-CLASS VARIANCE")
    print("-" * 80)
    print(f"   Gemma σ:      {var_comp['mean_gemma_std']:.6f}")
    print(f"   Vault σ:      {var_comp['mean_vault_std']:.6f}")
    print(f"   Ratio:        {var_comp['ratio']:.4f}")
    print(f"   p-value:      {var_comp['p_value']:.6f}")
    
    if var_comp['ratio'] > 1.5:
        print(f"   ✓ STRONG EFFECT: Vault Gemma has significantly wider variance")
    elif var_comp['ratio'] > 1.1:
        print(f"   ~ MODERATE EFFECT: Vault Gemma has moderately wider variance")
    else:
        print(f"   ✗ NO EFFECT: Similar variance between models")
    
    print(f"\n2. HIERARCHICAL ORTHOGONALITY")
    print("-" * 80)
    print(f"   Gemma |cos|:  {ortho_comp['mean_gemma_ortho']:.6f}")
    print(f"   Vault |cos|:  {ortho_comp['mean_vault_ortho']:.6f}")
    print(f"   Difference:   {ortho_comp['mean_vault_ortho'] - ortho_comp['mean_gemma_ortho']:.6f}")
    print(f"   p-value:      {ortho_comp['p_value']:.6f}")
    
    if ortho_comp['mean_vault_ortho'] < ortho_comp['mean_gemma_ortho'] * 0.9:
        print(f"   ✓ IMPROVEMENT: Vault Gemma has better orthogonality")
    elif ortho_comp['mean_vault_ortho'] > ortho_comp['mean_gemma_ortho'] * 1.1:
        print(f"   ✗ DEGRADATION: Vault Gemma has worse orthogonality")
    else:
        print(f"   ~ PRESERVED: Similar orthogonality between models")

print(f"\n{'='*80}")
print("OVERALL CONCLUSIONS")
print("="*80)
print("\nDifferential Privacy (Vault Gemma) Impact:")
print("\n1. Category Boundary Certainty (Intra-Class Variance):")
for dataset in DATASETS:
    ratio = variance_comparisons[dataset]['ratio']
    effect = "INCREASES" if ratio > 1.2 else "maintains"
    print(f"   - {dataset.upper()}: {effect} uncertainty (ratio={ratio:.3f})")

print("\n2. Hierarchical Structure Preservation (Orthogonality):")
for dataset in DATASETS:
    g_ortho = orthogonality_comparisons[dataset]['mean_gemma_ortho']
    v_ortho = orthogonality_comparisons[dataset]['mean_vault_ortho']
    if v_ortho < g_ortho * 0.95:
        effect = "IMPROVES"
    elif v_ortho > g_ortho * 1.05:
        effect = "DEGRADES"
    else:
        effect = "PRESERVES"
    print(f"   - {dataset.upper()}: {effect} hierarchical structure")

print("\n" + "="*80)

## 8. Save All Results

In [ ]:
# Prepare comprehensive results dictionary
comprehensive_results = {
    'models': {
        'gemma': GEMMA_MODEL,
        'vault_gemma': VAULT_GEMMA_MODEL
    },
    'datasets': DATASETS,
    'gemma_results': {},
    'vault_results': {},
    'comparisons': {
        'variance': variance_comparisons,
        'orthogonality': orthogonality_comparisons
    }
}

# Save simplified results (without heavy tensors)
for dataset in DATASETS:
    comprehensive_results['gemma_results'][dataset] = {
        'categories': gemma_results[dataset]['categories'],
        'summary': gemma_results[dataset]['summary'],
        'projection_stats': {k: {kk: vv for kk, vv in v.items() if kk != 'projections'}
                           for k, v in gemma_results[dataset]['projection_stats'].items()},
        'hierarchical_stats': gemma_results[dataset]['hierarchical_stats']
    }
    
    comprehensive_results['vault_results'][dataset] = {
        'categories': vault_results[dataset]['categories'],
        'summary': vault_results[dataset]['summary'],
        'projection_stats': {k: {kk: vv for kk, vv in v.items() if kk != 'projections'}
                           for k, v in vault_results[dataset]['projection_stats'].items()},
        'hierarchical_stats': vault_results[dataset]['hierarchical_stats']
    }

# Save to pickle
with open('comprehensive_wordnet_results.pkl', 'wb') as f:
    pickle.dump(comprehensive_results, f)

print("✓ Results saved to: comprehensive_wordnet_results.pkl")

# Also save a JSON version (without numpy arrays) for easier inspection
import json

json_results = {
    'models': comprehensive_results['models'],
    'datasets': comprehensive_results['datasets'],
    'summary': {}
}

for dataset in DATASETS:
    json_results['summary'][dataset] = {
        'gemma': gemma_results[dataset]['summary'],
        'vault': vault_results[dataset]['summary'],
        'variance_comparison': {
            'ratio': float(variance_comparisons[dataset]['ratio']),
            'p_value': float(variance_comparisons[dataset]['p_value'])
        },
        'orthogonality_comparison': {
            'gemma_avg': float(orthogonality_comparisons[dataset]['mean_gemma_ortho']),
            'vault_avg': float(orthogonality_comparisons[dataset]['mean_vault_ortho']),
            'p_value': float(orthogonality_comparisons[dataset]['p_value'])
        }
    }

with open('comprehensive_wordnet_results_summary.json', 'w') as f:
    json.dump(json_results, f, indent=2)

print("✓ Summary saved to: comprehensive_wordnet_results_summary.json")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  1. comprehensive_wordnet_results.pkl (full results)")
print("  2. comprehensive_wordnet_results_summary.json (summary)")
print("  3. wordnet_intra_class_variance_comparison.png")
print("  4. wordnet_hierarchical_orthogonality_comparison.png")
print("  5. wordnet_summary_comparison.png")
print("\n" + "="*80)

## References

This analysis is based on:
- **Paper**: "THE GEOMETRY OF CATEGORICAL AND HIERARCHICAL CONCEPTS IN LARGE LANGUAGE MODELS" (https://arxiv.org/abs/2406.01506)
- **Key Theorems**:
  - Theorem 4: Characterizes the projection distribution of words onto concept vectors
  - Hierarchical orthogonality principle: Parent-child relationships encoded as orthogonal directions
- **Datasets**: WordNet noun and verb synset hierarchies

## Next Steps

1. Investigate specific categories with largest variance differences
2. Analyze deeper hierarchical levels (grandparent-parent-child)
3. Test with different model sizes and DP parameters
4. Examine per-category variance to identify which semantic domains are most affected by DP